In [3]:
import requests
import pandas as pd
from datetime import datetime
import textwrap


# =========================
# CLEAN DISPLAY SETTINGS
# =========================
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 40)


# =========================
# CONVERT DATE
# =========================
def convert_date(ts):
    try:
        return datetime.fromtimestamp(ts).strftime("%Y-%m-%d")
    except:
        return "N/A"


# =========================
# CLEAN TITLE (IMPORTANT FIX)
# =========================
def clean_title(title, width=45):
    if not title:
        return "N/A"
    return textwrap.fill(title, width=width)


# =========================
# FETCH JOBS
# =========================
def fetch_jobs():

    url = "https://www.arbeitnow.com/api/job-board-api"
    data = requests.get(url).json()["data"]

    jobs = []

    for job in data:

        title = job.get("title", "")

        # FILTER ONLY DATA JOB ROLES
        if not any(x in title.lower() for x in [
            "data analyst",
            "business analyst",
            "data scientist",
            "business intelligence",
            "bi"
        ]):
            continue

        posted = job.get("created_at")
        if isinstance(posted, (int, float)):
            posted = convert_date(posted)

        jobs.append({
            "Title": clean_title(title),
            "Company": job.get("company_name"),
            "Location": job.get("location"),
            "Remote": job.get("remote"),
            "Posted": posted
        })

    return jobs


# =========================
# BUILD DATAFRAME
# =========================
def build_df(jobs):

    df = pd.DataFrame(jobs)

    if df.empty:
        return df

    df = df.sort_values(by="Posted", ascending=False)

    return df


# =========================
# MAIN
# =========================
def main():

    jobs = fetch_jobs()
    df = build_df(jobs)

    print("\n==============================")
    print("DATA / BI JOBS (CLEAN VIEW)")
    print("==============================\n")

    if df.empty:
        print("No matching jobs found.")
    else:
        print(df.to_string(index=False))


if __name__ == "__main__":
    main()


DATA / BI JOBS (CLEAN VIEW)

                                                                                  Title                   Company      Location  Remote     Posted
Steuerfachangestellter/Steuerfachwirt/Bilanzb\nuchhalter (m/w/d) in Voll- oder Teilzeit KSP Stübben & Partner mbB    Düsseldorf   False 2026-05-06
Steuerfachangestellter/Steuerfachwirt/Bilanzb\nuchhalter (m/w/d) in Voll- oder Teilzeit KSP Stübben & Partner mbB        Munich   False 2026-05-06
                               Ausbildung Fachinformatiker für\nSystemintegration m|w|d                abat+ GmbH Sankt Ingbert   False 2026-05-06
               Weiterbildung: Data Science & Business\nAnalytics (IHK) (m/w/d) - Remote      DataSmart Point GmbH        Berlin    True 2026-05-06
                  Karrierestart in Data & AI - IHK Data Analyst\n(m/w/d) - 100 % Online      DataSmart Point GmbH        Berlin    True 2026-05-06
